# Phase 3c — Few-Shot Domain Adaptation: Córdoba

**Environment**: Colab A100  
**Objective**: fine-tune the FPN decoder on 100 Córdoba patches to reduce domain gap  
**Base**: model trained in 07_prithvi.ipynb (epochs 1–40, Corrientes IoU ≥ 0.42)  
**Strategy**: Prithvi encoder frozen, only FPN decoder is trained  

## Instructions
1. `Runtime → Change runtime type → A100 GPU`
2. `Runtime → Run All` — the first cell installs dependencies and **restarts the kernel automatically**
3. Click `Run All` again after restart
4. Fine-tuning takes ~5 min on A100

In [ ]:
import subprocess, sys, os

needs_install = False
try:
    import numpy as np
    assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0), \
        f'numpy {np.__version__} < 2.0'
    from terratorch.registry import BACKBONE_REGISTRY
    print(f'OK numpy={np.__version__} | terratorch ready — skipping install.')
except Exception as e:
    needs_install = True
    print(f'Installing dependencies ({e})...')

if needs_install:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy==2.0.2', '--force-reinstall'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'terratorch', 'einops', 'timm',
                    'segmentation-models-pytorch'], check=True)
    print('\nInstall complete. Restarting kernel...')
    os.kill(os.getpid(), 9)

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

print(f'IN_COLAB = {IN_COLAB}')

In [ ]:
import os, random, subprocess, warnings
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import roc_auc_score
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'numpy  : {np.__version__}')
print(f'torch  : {torch.__version__}')
print(f'device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('G:/Mon Drive/GeoAI/wildfire-spread')

CKPT_DIR = BASE / 'models'
RESULTS  = BASE / 'results'

if IN_COLAB:
    LOCAL = Path('/content/cordoba')
    LOCAL.mkdir(exist_ok=True)
    if not (LOCAL / 'images').exists():
        print('Copiando patches de Córdoba a /content/...')
        subprocess.run(['cp', '-r', str(BASE / 'data/cordoba/patches/images'),      str(LOCAL)], check=True)
        subprocess.run(['cp', '-r', str(BASE / 'data/cordoba/patches/masks_dnbr'),  str(LOCAL)], check=True)
        print('Copia completada.')
    else:
        print('Patches de Córdoba ya en /content/')
    IMG_DIR  = LOCAL / 'images'
    MASK_DIR = LOCAL / 'masks_dnbr'
else:
    IMG_DIR  = BASE / 'data/cordoba/patches/images'
    MASK_DIR = BASE / 'data/cordoba/patches/masks_dnbr'

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE      = 224
BATCH_SIZE    = 8
N_FINETUNE    = 100       # patches used for fine-tuning (stratified)
EPOCHS_FT     = 15        # fine-tuning epochs
LR_FT         = 5e-5      # lower LR than initial training
FIRE_WEIGHT   = 3.0       # Córdoba has 63.7% positives — lower weight
THRESHOLD     = 0.5
MODEL_NAME    = 'prithvi_burn_scar'
CKPT_BASE     = CKPT_DIR / f'best_{MODEL_NAME}_wildfire.pth'       # modelo de 07
CKPT_FT       = CKPT_DIR / f'best_{MODEL_NAME}_cordoba_ft.pth'    # modelo fine-tuned

img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
assert len(img_paths) == len(mask_paths) and len(img_paths) > 0, \
    f'ERROR: {len(img_paths)} imgs, {len(mask_paths)} masks'
print(f'Patches Córdoba: {len(img_paths)}')
assert CKPT_BASE.exists(), f'Base checkpoint not found: {CKPT_BASE}'
print(f'Checkpoint base: {CKPT_BASE.name}')

In [ ]:
# ── Prithvi normalization (same stats as 07_prithvi.ipynb) ──────────────
PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449],
                         dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420],
                         dtype=np.float32)
BAND_IDX = [0, 1, 2, 4, 5, 6]   # B02, B03, B04, B8A, B11, B12


class WildfireDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.augment    = augment
        self._t = (256 - IMG_SIZE) // 2

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img  = np.load(self.img_paths[idx]).astype(np.float32)
        mask = np.load(self.mask_paths[idx]).astype(np.float32)
        img = img[BAND_IDX] / 10000.0
        img = (img - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        t = self._t
        img  = img[:, t:t+IMG_SIZE, t:t+IMG_SIZE]
        mask = mask[t:t+IMG_SIZE, t:t+IMG_SIZE]
        if self.augment:
            if random.random() > 0.5: img, mask = np.flip(img, 2).copy(), np.flip(mask, 1).copy()
            if random.random() > 0.5: img, mask = np.flip(img, 1).copy(), np.flip(mask, 0).copy()
            if random.random() > 0.5:
                k = random.randint(1, 3)
                img  = np.rot90(img,  k, (1, 2)).copy()
                mask = np.rot90(mask, k, (0, 1)).copy()
        return torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)

print('WildfireDataset defined.')

In [ ]:
# ── Architecture (identical to 07_prithvi.ipynb) ────────────────────────────────
class PrithviNeck(nn.Module):
    def __init__(self, n_side=14):
        super().__init__()
        self.n_side = n_side
    def forward(self, layers_out):
        x = layers_out[-1][:, 1:, :]
        B, L, D = x.shape
        return x.permute(0,2,1).reshape(B, D, self.n_side, self.n_side)


class FPNDecoder(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModel(nn.Module):
    def __init__(self, backbone, embed_dim=768, n_side=14):
        super().__init__()
        self.backbone = backbone
        self.neck     = PrithviNeck(n_side)
        self.decoder  = FPNDecoder(embed_dim)
    def forward(self, x):
        feats = self.backbone(x.unsqueeze(2))
        sp    = self.neck(feats)
        return self.decoder(sp)

print('Architecture defined.')

In [ ]:
# ── Load Prithvi + weights from 07_prithvi.ipynb ────────────────────────────────
from terratorch.registry import BACKBONE_REGISTRY

print('Loading prithvi_eo_v1_100...')
backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100').to(DEVICE)
backbone.eval()

with torch.no_grad():
    _x   = torch.zeros(1, 6, 224, 224, device=DEVICE).unsqueeze(2)
    _out = backbone(_x)
OUT_DIM = _out[-1].shape[-1]
N_SIDE  = int((_out[-1].shape[1] - 1)**0.5)
del _x, _out; torch.cuda.empty_cache()

for p in backbone.parameters():
    p.requires_grad = False

model = PrithviSegModel(backbone, embed_dim=OUT_DIM, n_side=N_SIDE).to(DEVICE)
model.load_state_dict(torch.load(CKPT_BASE, map_location=DEVICE))
print(f'Weights loaded from: {CKPT_BASE.name}')

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'{n_total/1e6:.1f}M total | {n_train/1e6:.2f}M trainable (neck+decoder)')

In [ ]:
# ── Split: 100 fine-tune patches / resto test ─────────────────────────────────
print('Scanning fire flags...')
fire_flags = np.array([
    np.load(p, mmap_mode='r').max() > 0 for p in tqdm(mask_paths)
], dtype=np.int32)

indices = np.arange(len(img_paths))

# Separar 100 patches para fine-tuning (estratificados)
ft_idx, test_idx = train_test_split(
    indices,
    train_size=N_FINETUNE,
    stratify=fire_flags,
    random_state=SEED
)

print(f'Fine-tune : {len(ft_idx):>5} patches ({fire_flags[ft_idx].sum()} positive, '
      f'{fire_flags[ft_idx].mean()*100:.1f}%)')
print(f'Test      : {len(test_idx):>5} patches ({fire_flags[test_idx].sum()} positive, '
      f'{fire_flags[test_idx].mean()*100:.1f}%)')

ft_ds   = WildfireDataset([img_paths[i] for i in ft_idx],   [mask_paths[i] for i in ft_idx],   augment=True)
test_ds = WildfireDataset([img_paths[i] for i in test_idx], [mask_paths[i] for i in test_idx], augment=False)

ft_loader   = DataLoader(ft_ds,   batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Fine-tune batches: {len(ft_loader)}  |  Test batches: {len(test_loader)}')

In [ ]:
# ── Evaluación ANTES del fine-tuning (baseline Córdoba) ──────────────────────
def evaluate(model, loader, threshold=THRESHOLD):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc='Evaluating', leave=False):
            probs    = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
            masks_np = masks.squeeze(1).cpu().numpy()
            all_probs.append(probs.reshape(-1))
            all_targets.append(masks_np.reshape(-1))
    all_probs   = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets).astype(np.int32)
    all_preds   = (all_probs > threshold).astype(np.int32)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, pos_label=1, average='binary', zero_division=0)
    tp = int(((all_preds==1)&(all_targets==1)).sum())
    fp = int(((all_preds==1)&(all_targets==0)).sum())
    fn = int(((all_preds==0)&(all_targets==1)).sum())
    iou = tp / (tp + fp + fn + 1e-6)
    try:
        auc = roc_auc_score(all_targets, all_probs)
    except Exception:
        auc = float('nan')
    return dict(iou=iou, precision=prec, recall=rec, f1=f1, auc=auc)


print('Evaluating BASE model (no fine-tuning) on Córdoba test set...')
results_before = evaluate(model, test_loader)
print(f'  IoU       : {results_before["iou"]:.4f}')
print(f'  Recall    : {results_before["recall"]:.4f}')
print(f'  Precision : {results_before["precision"]:.4f}')
print(f'  AUC-ROC   : {results_before["auc"]:.4f}')

In [ ]:
# ── Decoder fine-tuning on 100 Córdoba patches ─────────────────────
dice_loss  = DiceLoss(mode='binary', from_logits=True)
focal_loss = FocalLoss(mode='binary', gamma=2.0, alpha=0.75)

def criterion(pred, target):
    return dice_loss(pred, target) + focal_loss(pred, target)

# Solo entrenar neck + decoder (encoder sigue congelado)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LR_FT, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS_FT, eta_min=1e-7)

best_iou_ft  = 0.0
ft_losses, ft_ious = [], []

print(f'Fine-tuning decoder — {EPOCHS_FT} epochs | LR={LR_FT} | {N_FINETUNE} Córdoba patches')
for epoch in range(1, EPOCHS_FT + 1):
    model.train()
    model.backbone.eval()   # encoder always frozen
    ep_loss = 0.0
    for imgs, masks in tqdm(ft_loader, desc=f'FT {epoch:02d}/{EPOCHS_FT}', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward(); optimizer.step()
        ep_loss += loss.item()
    scheduler.step()
    ft_losses.append(ep_loss / len(ft_loader))

    # Validar sobre test set
    res = evaluate(model, test_loader)
    ft_ious.append(res['iou'])

    print(f'Epoch {epoch:02d} | Loss: {ft_losses[-1]:.4f} | IoU: {ft_ious[-1]:.4f}')

    if ft_ious[-1] > best_iou_ft:
        best_iou_ft = ft_ious[-1]
        torch.save(model.state_dict(), CKPT_FT)
        print(f'  → Saved (IoU: {best_iou_ft:.4f})')

print(f'\nBest IoU fine-tuned: {best_iou_ft:.4f}')

In [ ]:
# ── Evaluación DESPUÉS del fine-tuning ───────────────────────────────────────
model.load_state_dict(torch.load(CKPT_FT, map_location=DEVICE))
results_after = evaluate(model, test_loader)

print('\n' + '='*60)
print('  COMPARISON — Córdoba test set')
print('='*60)
print(f'  {"Métrica":<12} {"Base (07)":>12} {"Fine-tuned":>12} {"Mejora":>10}')
print('-'*60)
for key in ['iou', 'recall', 'precision', 'auc']:
    label = {'iou': 'IoU', 'recall': 'Recall', 'precision': 'Precision', 'auc': 'AUC-ROC'}[key]
    before = results_before[key]
    after  = results_after[key]
    delta  = after - before
    print(f'  {label:<12} {before:>12.4f} {after:>12.4f} {delta:>+10.4f}')
print('='*60)
print(f'\nFine-tuned model saved: {CKPT_FT.name}')

In [ ]:
# ── Curvas de fine-tuning ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, EPOCHS_FT+1), ft_losses, color='steelblue', label='Fine-tune loss')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss — fine-tuning Córdoba')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, EPOCHS_FT+1), ft_ious, color='green', label='IoU Córdoba')
axes[1].axhline(results_before['iou'], color='gray', ls=':', label=f'Base: {results_before["iou"]:.4f}')
axes[1].axhline(best_iou_ft,           color='red',  ls='--', label=f'Best FT: {best_iou_ft:.4f}')
axes[1].set(xlabel='Epoch', ylabel='IoU', title='IoU Córdoba — fine-tuning')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
out = RESULTS / 'cordoba_finetune_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Visualización — predicciones del modelo fine-tuned ───────────────────────
def norm_band(x, p=2):
    v = x[x > 0]
    if not len(v): return np.zeros_like(x)
    lo, hi = np.percentile(v, [p, 100-p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

model.eval()
sample_imgs, sample_masks, sample_probs = [], [], []
with torch.no_grad():
    for imgs, masks in test_loader:
        probs = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
        for b in range(len(imgs)):
            m = masks[b].squeeze().numpy()
            if m.max() > 0:
                sample_imgs.append(imgs[b].numpy())
                sample_masks.append(m)
                sample_probs.append(probs[b])
        if len(sample_imgs) >= 3:
            break

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for row in range(min(3, len(sample_imgs))):
    img_np  = sample_imgs[row]
    prob_np = sample_probs[row]
    mask_np = sample_masks[row]
    pred_b  = (prob_np > THRESHOLD).astype(np.float32)

    def dn(ch):
        return norm_band(img_np[ch] * PRITHVI_STD[ch] + PRITHVI_MEAN[ch])
    rgb = np.dstack([dn(2), dn(1), dn(0)])

    err = np.zeros((*mask_np.shape, 3))
    err[(pred_b==1)&(mask_np==1)] = [0.0, 0.8, 0.0]
    err[(pred_b==1)&(mask_np==0)] = [1.0, 0.5, 0.0]
    err[(pred_b==0)&(mask_np==1)] = [0.9, 0.1, 0.1]

    axes[row,0].imshow(rgb);                                    axes[row,0].axis('off')
    axes[row,1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1);  axes[row,1].axis('off')
    axes[row,2].imshow(prob_np, cmap='hot',  vmin=0, vmax=1);  axes[row,2].axis('off')
    axes[row,3].imshow(err);                                    axes[row,3].axis('off')

for ax, t in zip(axes[0], ['RGB', 'Ground truth (dNBR)', 'Prediction (prob)', 'TP/FP/FN']):
    ax.set_title(t, fontsize=10)

tp_p = mpatches.Patch(color=[0,.8,0], label='TP')
fp_p = mpatches.Patch(color=[1,.5,0], label='FP')
fn_p = mpatches.Patch(color=[.9,.1,.1], label='FN')
fig.legend(handles=[tp_p,fp_p,fn_p], loc='lower center', ncol=3, bbox_to_anchor=(0.5,-0.01))
plt.suptitle(f'Córdoba — Fine-tuned | IoU base: {results_before["iou"]:.4f} → FT: {best_iou_ft:.4f}',
             fontsize=12, y=1.01)
plt.tight_layout()
out = RESULTS / 'cordoba_finetune_predictions.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Resumen portfolio ─────────────────────────────────────────────────────────
print('='*65)
print('  PORTFOLIO SUMMARY — Few-Shot Domain Adaptation')
print('='*65)
print(f'  Modelo base  : Prithvi-100M + FPN (trained on Corrientes 2022)')
print(f'  Fine-tune    : {N_FINETUNE} Córdoba 2020 patches ({EPOCHS_FT} épocas)')
print(f'  Strategy     : encoder frozen, decoder only updated')
print('-'*65)
print(f'  {"Métrica":<12} {"Corrientes val":>14} {"Córdoba base":>14} {"Córdoba FT":>12}')
print('-'*65)
print(f'  {"IoU":<12} {0.4198:>14.4f} {results_before["iou"]:>14.4f} {best_iou_ft:>12.4f}')
print(f'  {"Recall":<12} {0.7303:>14.4f} {results_before["recall"]:>14.4f} {results_after["recall"]:>12.4f}')
print(f'  {"Precision":<12} {0.4968:>14.4f} {results_before["precision"]:>14.4f} {results_after["precision"]:>12.4f}')
print('='*65)